# Pipeline de Migração de Dados de RH — Apache Iceberg

Este notebook implementa a mesma pipeline de dados de RH da **TechCorp**, agora utilizando **PySpark** com **Apache Iceberg** como formato de tabela.

## Fonte de Dados
Os mesmos CSVs do sistema legado de RH:
- `funcionarios.csv`, `departamentos.csv`, `cargos.csv`, `folha_pagamento.csv`

## Por que Iceberg?
- Suporte a schema evolution sem reescrever dados
- Particionamento oculto (hidden partitioning)
- Suporte nativo a múltiplos engines (Spark, Flink, Trino, Hive)
- Snapshots e time travel via metadados JSON

## 1. Inicialização da SparkSession com Apache Iceberg
Configura o SparkSession com o catálogo Iceberg usando o backend local (HadoopCatalog).

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, current_timestamp, lit, when
from pyspark.sql.types import *

ICEBERG_WAREHOUSE = '/tmp/iceberg/warehouse'

# SparkSession com Iceberg — o pacote é carregado via --packages ou config
spark = (
    SparkSession.builder
    .appName('Pipeline_RH_Iceberg')
    # Extensão SQL do Iceberg
    .config('spark.sql.extensions', 'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions')
    # Catálogo nomeado 'local' usando HadoopCatalog (sem Hive Metastore)
    .config('spark.sql.catalog.local', 'org.apache.iceberg.spark.SparkCatalog')
    .config('spark.sql.catalog.local.type', 'hadoop')
    .config('spark.sql.catalog.local.warehouse', ICEBERG_WAREHOUSE)
    # Adiciona o pacote Iceberg via Maven
    .config('spark.jars.packages', 'org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0')
    .getOrCreate()
)

spark.sparkContext.setLogLevel('WARN')
print(f'✅ SparkSession com Iceberg iniciada — versão Spark: {spark.version}')

## 2. Criação do Namespace (Database) Iceberg
No Iceberg, as tabelas ficam dentro de namespaces — equivalentes a schemas/databases.

In [ ]:
# Cria o namespace 'rh' no catálogo 'local'
spark.sql('CREATE NAMESPACE IF NOT EXISTS local.rh')
spark.sql('SHOW NAMESPACES IN local').show()
print('✅ Namespace local.rh criado!')

## 3. Leitura dos Dados da Fonte (CSV)
Carrega os arquivos CSV do sistema legado.

In [ ]:
DATA_PATH = '../dados'

# Lê os CSVs com detecção automática de tipos
df_funcionarios  = spark.read.csv(f'{DATA_PATH}/funcionarios.csv',  header=True, inferSchema=True)
df_departamentos = spark.read.csv(f'{DATA_PATH}/departamentos.csv', header=True, inferSchema=True)
df_cargos        = spark.read.csv(f'{DATA_PATH}/cargos.csv',        header=True, inferSchema=True)
df_folha         = spark.read.csv(f'{DATA_PATH}/folha_pagamento.csv', header=True, inferSchema=True)

print(f'👥 Funcionários:  {df_funcionarios.count()} registros')
print(f'🏢 Departamentos: {df_departamentos.count()} registros')
print(f'📋 Cargos:        {df_cargos.count()} registros')
print(f'💰 Folha:         {df_folha.count()} registros')

## 4. Transformação dos Dados
Aplica tipagem correta e adiciona colunas de controle de pipeline.

In [ ]:
# Transforma colunas de data e adiciona metadados de rastreamento
df_func_clean = (
    df_funcionarios
    .withColumn('data_nascimento', to_date(col('data_nascimento'), 'yyyy-MM-dd'))
    .withColumn('data_admissao',   to_date(col('data_admissao'),   'yyyy-MM-dd'))
    .withColumn('salario',         col('salario').cast(DoubleType()))
    .withColumn('pipeline_ts',     current_timestamp())  # timestamp da execução da pipeline
    .withColumn('source',          lit('CSV_LEGADO'))
)

df_folha_clean = (
    df_folha
    .withColumn('data_pagamento', to_date(col('data_pagamento'), 'yyyy-MM-dd'))
    .withColumn('pipeline_ts',   current_timestamp())
    .withColumn('source',        lit('CSV_LEGADO'))
)

df_dep_clean   = df_departamentos.withColumn('pipeline_ts', current_timestamp())
df_cargo_clean = df_cargos.withColumn('pipeline_ts', current_timestamp())

print('✅ Transformações aplicadas!')

## 5. Criação das Tabelas Iceberg via DDL
Define as tabelas com schema explícito usando SQL Iceberg.

In [ ]:
# DDL: cria tabela de departamentos no formato Iceberg
spark.sql("""
    CREATE TABLE IF NOT EXISTS local.rh.departamentos (
        departamento_id   INT,
        nome_departamento STRING,
        sigla             STRING,
        centro_custo      STRING,
        gerente_id        INT,
        orcamento_anual   DOUBLE,
        pipeline_ts       TIMESTAMP
    )
    USING iceberg
""")
print('🏢 Tabela local.rh.departamentos criada!')

# DDL: cria tabela de cargos
spark.sql("""
    CREATE TABLE IF NOT EXISTS local.rh.cargos (
        cargo_id           INT,
        titulo_cargo       STRING,
        nivel              STRING,
        faixa_salarial_min DOUBLE,
        faixa_salarial_max DOUBLE,
        pipeline_ts        TIMESTAMP
    )
    USING iceberg
""")
print('📋 Tabela local.rh.cargos criada!')

# DDL: cria tabela de funcionários com particionamento por status
spark.sql("""
    CREATE TABLE IF NOT EXISTS local.rh.funcionarios (
        funcionario_id  INT,
        nome            STRING,
        email           STRING,
        cpf             STRING,
        data_nascimento DATE,
        data_admissao   DATE,
        departamento_id INT,
        cargo_id        INT,
        salario         DOUBLE,
        status          STRING,
        pipeline_ts     TIMESTAMP,
        source          STRING
    )
    USING iceberg
    PARTITIONED BY (status)
""")
print('👥 Tabela local.rh.funcionarios criada (particionada por status)!')

# DDL: cria tabela de folha com particionamento por competencia
spark.sql("""
    CREATE TABLE IF NOT EXISTS local.rh.folha_pagamento (
        folha_id            INT,
        funcionario_id      INT,
        competencia         STRING,
        salario_bruto       DOUBLE,
        desconto_inss       DOUBLE,
        desconto_irrf       DOUBLE,
        desconto_plano_saude DOUBLE,
        bonus               DOUBLE,
        salario_liquido     DOUBLE,
        data_pagamento      DATE,
        pipeline_ts         TIMESTAMP,
        source              STRING
    )
    USING iceberg
    PARTITIONED BY (competencia)
""")
print('💰 Tabela local.rh.folha_pagamento criada (particionada por competência)!')

## 6. INSERT — Carga Inicial dos Dados
Insere os dados transformados nas tabelas Iceberg.

In [ ]:
# -------------------------------------------------------
# INSERT: Insere os dados das tabelas de apoio
# -------------------------------------------------------
df_dep_clean.writeTo('local.rh.departamentos').append()
print('✅ Departamentos inseridos!')

df_cargo_clean.writeTo('local.rh.cargos').append()
print('✅ Cargos inseridos!')

# Seleciona apenas as colunas definidas no DDL
df_func_clean.select(
    'funcionario_id','nome','email','cpf',
    'data_nascimento','data_admissao','departamento_id',
    'cargo_id','salario','status','pipeline_ts','source'
).writeTo('local.rh.funcionarios').append()
print('✅ Funcionários inseridos!')

df_folha_clean.select(
    'folha_id','funcionario_id','competencia','salario_bruto',
    'desconto_inss','desconto_irrf','desconto_plano_saude',
    'bonus','salario_liquido','data_pagamento','pipeline_ts','source'
).writeTo('local.rh.folha_pagamento').append()
print('✅ Folha de pagamento inserida!')

# Verifica totais
print(f'\nTotal funcionários no Iceberg: {spark.table("local.rh.funcionarios").count()}')

In [ ]:
# Confere os dados inseridos
spark.sql('SELECT funcionario_id, nome, cargo_id, salario, status FROM local.rh.funcionarios ORDER BY 1').show()

## 7. INSERT — Novos Registros via SQL
O Iceberg suporta INSERT INTO via SQL, assim como tabelas relacionais.

In [ ]:
# INSERT via SQL — adiciona novos funcionários contratados em 2024
spark.sql("""
    INSERT INTO local.rh.funcionarios VALUES
    (16, 'Renata Gomes', 'renata.gomes@techcorp.com', '666.777.888-99',
     DATE '1998-04-12', DATE '2024-03-01', 1, 1, 5000.0, 'ATIVO',
     current_timestamp(), 'SISTEMA_RH_2024'),
    (17, 'Gustavo Prado', 'gustavo.prado@techcorp.com', '777.888.999-00',
     DATE '1990-09-27', DATE '2024-03-15', 3, 4, 10500.0, 'ATIVO',
     current_timestamp(), 'SISTEMA_RH_2024')
""")

total = spark.sql('SELECT COUNT(*) as total FROM local.rh.funcionarios').collect()[0]['total']
print(f'✅ Novos funcionários inseridos via SQL! Total: {total}')

## 8. UPDATE — Atualizando Registros no Iceberg
O Iceberg suporta UPDATE direto via SQL desde a versão 0.14 com Spark 3.x.

In [ ]:
# -------------------------------------------------------
# UPDATE: Reajuste salarial de 12% para analistas plenos
# -------------------------------------------------------
spark.sql("""
    UPDATE local.rh.funcionarios
    SET salario     = salario * 1.12,
        pipeline_ts = current_timestamp()
    WHERE cargo_id = 2
    AND   status   = 'ATIVO'
""")

print('✅ UPDATE: Analistas Plenos receberam reajuste de 12%!')
spark.sql("""
    SELECT nome, cargo_id, salario, status
    FROM local.rh.funcionarios
    WHERE cargo_id = 2
""").show()

In [ ]:
# UPDATE: marca funcionários inativos como DESLIGADO
spark.sql("""
    UPDATE local.rh.funcionarios
    SET status      = 'DESLIGADO',
        pipeline_ts = current_timestamp()
    WHERE status = 'INATIVO'
""")

print('✅ UPDATE: funcionários INATIVO → DESLIGADO')
spark.sql("SELECT nome, status FROM local.rh.funcionarios WHERE status = 'DESLIGADO'").show()

## 9. DELETE — Removendo Registros no Iceberg
DELETE seletivo com predicado SQL. O Iceberg registra um novo snapshot sem modificar arquivos antigos.

In [ ]:
antes = spark.sql('SELECT COUNT(*) as n FROM local.rh.funcionarios').collect()[0]['n']
print(f'Registros antes do DELETE: {antes}')

# -------------------------------------------------------
# DELETE: Remove funcionários com status DESLIGADO
# -------------------------------------------------------
spark.sql("""
    DELETE FROM local.rh.funcionarios
    WHERE status = 'DESLIGADO'
""")

depois = spark.sql('SELECT COUNT(*) as n FROM local.rh.funcionarios').collect()[0]['n']
print(f'Registros após o DELETE:   {depois}')
print(f'🗑️  {antes - depois} registro(s) removido(s)!')

In [ ]:
# DELETE em folha de pagamento — remove registros de fonte legada
spark.sql("""
    DELETE FROM local.rh.folha_pagamento
    WHERE source = 'CSV_LEGADO'
    AND   data_pagamento < DATE '2024-01-01'
""")
print('✅ DELETE: registros de folha anteriores a 2024 removidos')

## 10. MERGE INTO — UPSERT no Iceberg
O `MERGE INTO` é o comando mais completo para sincronização de dados no Iceberg.

In [ ]:
from pyspark.sql import Row
from datetime import date

# Dados incrementais chegando da API do sistema de RH online
dados_novos = [
    Row(funcionario_id=1,  nome='Ana Paula Silva',  email='ana.paula@techcorp.com',
        cpf='123.456.789-01', data_nascimento=date(1990, 3, 15),
        data_admissao=date(2018, 6, 1), departamento_id=1, cargo_id=3,
        salario=9500.0, status='ATIVO', pipeline_ts=None, source='API_RH'),
    Row(funcionario_id=18, nome='Sofia Barros', email='sofia.barros@techcorp.com',
        cpf='888.999.000-11', data_nascimento=date(2000, 6, 18),
        data_admissao=date(2024, 4, 1), departamento_id=4, cargo_id=1,
        salario=4800.0, status='ATIVO', pipeline_ts=None, source='API_RH'),
]

df_novos = spark.createDataFrame(dados_novos).withColumn('pipeline_ts', current_timestamp())

# Registra como view temporária para usar no MERGE SQL
df_novos.createOrReplaceTempView('source_rh')

# MERGE INTO: Iceberg suporta matched/not-matched com múltiplas cláusulas
spark.sql("""
    MERGE INTO local.rh.funcionarios AS target
    USING source_rh AS source
    ON target.funcionario_id = source.funcionario_id
    WHEN MATCHED THEN
        UPDATE SET
            target.salario      = source.salario,
            target.email        = source.email,
            target.pipeline_ts  = source.pipeline_ts
    WHEN NOT MATCHED THEN
        INSERT *
""")

total = spark.sql('SELECT COUNT(*) as n FROM local.rh.funcionarios').collect()[0]['n']
print(f'✅ MERGE INTO executado! Total: {total} registros')

## 11. Time Travel e Snapshots no Iceberg
O Iceberg registra metadados de cada snapshot, permitindo consulta histórica.

In [ ]:
# Lista todos os snapshots da tabela
print('=== Snapshots da tabela local.rh.funcionarios ===')
spark.sql("""
    SELECT snapshot_id, committed_at, operation, summary
    FROM local.rh.funcionarios.snapshots
""").show(truncate=False)

In [ ]:
# Time Travel: lê os dados do primeiro snapshot (snapshot_id do SELECT acima)
snapshots = spark.sql('SELECT snapshot_id FROM local.rh.funcionarios.snapshots').collect()

if snapshots:
    first_snapshot_id = snapshots[0]['snapshot_id']
    df_historico = spark.read.option('snapshot-id', first_snapshot_id) \
        .table('local.rh.funcionarios')
    print(f'📸 Snapshot inicial ({first_snapshot_id}): {df_historico.count()} registros')
    df_historico.select('nome', 'salario', 'status').show(5)

## 12. Schema Evolution — Evolução de Schema
Um diferencial do Iceberg: adicionar colunas sem reescrever dados existentes.

In [ ]:
# Adiciona coluna nova sem reescrever a tabela inteira
spark.sql("""
    ALTER TABLE local.rh.funcionarios
    ADD COLUMN data_desligamento DATE
""")

print('✅ Coluna data_desligamento adicionada!')
spark.sql('DESCRIBE local.rh.funcionarios').show(truncate=False)

## 13. Consulta Analítica Final
Relatório consolidado de headcount e folha por departamento.

In [ ]:
# Relatório final: headcount, salário médio e folha por departamento
spark.sql("""
    SELECT
        d.nome_departamento,
        d.sigla,
        COUNT(f.funcionario_id) AS headcount,
        ROUND(AVG(f.salario), 2)  AS salario_medio,
        ROUND(SUM(f.salario), 2)  AS folha_total
    FROM local.rh.funcionarios f
    JOIN local.rh.departamentos d
      ON f.departamento_id = d.departamento_id
    WHERE f.status = 'ATIVO'
    GROUP BY d.nome_departamento, d.sigla
    ORDER BY folha_total DESC
""").show()

print('\n✅ Pipeline Iceberg concluída com sucesso!')

In [ ]:
# Encerra a SparkSession
spark.stop()
print('🔴 SparkSession encerrada.')